# Purpose
- Unzip data
- Get Label csv files

# (m)Parameter list

##### Params

In [ ]:
PROJECT_FOLDER = r'/content/drive/MyDrive/Franklin_Project_2024'
BASE_URL = "https://dl.cv.ethz.ch/bdd100k/data"
ZIP_FILES = [
    "100k_images_val.zip",
    "100k_images_train.zip",
    "100k_images_test.zip",
    "bdd100k_det_20_labels_trainval.zip"
]

DOWNLOADED_ZIP_FILES = [
    f"{PROJECT_FOLDER}/data/zip/100k_images_val.zip",
    f"{PROJECT_FOLDER}/data/zip/100k_images_train.zip",
    f"{PROJECT_FOLDER}/data/zip/100k_images_test.zip",
    f"{PROJECT_FOLDER}/data/zip/bdd100k_det_20_labels_trainval.zip"
]

ANALYSIS_FOLDER = f"{PROJECT_FOLDER}/data/analysis"
LABEL_TRAIN_FILE = f'{PROJECT_FOLDER}/data/bdd100k/bdd100k/labels/det_20/det_train.json'
LABEL_VAL_FILE = f'{PROJECT_FOLDER}/data/bdd100k/bdd100k/labels/det_20/det_val.json'

BATCH_SIZE, NUM_WORKERS = 32, 4


##### Libs

In [ ]:
import os
import requests
import zipfile
import csv
from collections import Counter
import pandas as pd
import os
import cv2
import numpy as np
import torch
from torch.utils.data import Dataset, DataLoader
import matplotlib.pyplot as plt
from tqdm import tqdm

# Download

In [ ]:
import os
import requests

def download_files(base_url=None, files=[], project_folder=None):
    # Define the dowhttps://research.google.com/colaboratory/faq.html#drive-timeoutnload folder

    if not base_url:
        base_url = BASE_URL

    if not files:
        files = ZIP_FILES

    if not project_folder:
        project_folder = PROJECT_FOLDER

    download_folder = os.path.join(project_folder, 'data', 'zip')

    # Create the download folder if it doesn't exist
    if not os.path.exists(download_folder):
        os.makedirs(download_folder)

    # Loop through each file and download it
    for file in files:
        file_url = f"{base_url}/{file}"
        file_path = os.path.join(download_folder, file)

        # Download the file
        print(f"Downloading {file}...")
        response = requests.get(file_url, stream=True)
        response.raise_for_status()  # Check for request errors

        # Write the content to a file
        with open(file_path, 'wb') as f:
            for chunk in response.iter_content(chunk_size=8192):
                f.write(chunk)

        print(f"{file} downloaded successfully.")

    print("All files downloaded.")


In [ ]:
if 1 == 11:
    download_files()

# Unzip

In [ ]:
import os
import zipfile

def unzip_files(downloaded_zip_files=None, project_folder=None):
    # Define the unzipped folder
    if not project_folder:
        project_folder = PROJECT_FOLDER
    unzipped_folder = os.path.join(project_folder, 'data', 'unzipped')

    if not downloaded_zip_files:
        downloaded_zip_files = DOWNLOADED_ZIP_FILES

    # Create the unzipped folder if it doesn't exist
    if not os.path.exists(unzipped_folder):
        os.makedirs(unzipped_folder)

    # Loop through each zip file and unzip it
    for zip_file in downloaded_zip_files:
        with zipfile.ZipFile(zip_file, 'r') as zip_ref:
            # Extract all the contents into the unzipped folder
            print(f"Unzipping {os.path.basename(zip_file)}...")
            zip_ref.extractall(unzipped_folder)
            print(f"{os.path.basename(zip_file)} unzipped successfully.")

    print("All files unzipped.")




In [ ]:
if 11 == 111:
    unzip_files()

Unzipping 100k_images_val.zip...
100k_images_val.zip unzipped successfully.
Unzipping 100k_images_train.zip...
100k_images_train.zip unzipped successfully.
Unzipping 100k_images_test.zip...
100k_images_test.zip unzipped successfully.
Unzipping bdd100k_det_20_labels_trainval.zip...
bdd100k_det_20_labels_trainval.zip unzipped successfully.
All files unzipped.


# (m)Inventory

In [ ]:
import os
import csv

def list_images(folder_path=None, output_csv=None):
    image_list = []

    # Walk through all subdirectories and files
    if not folder_path:
        folder_path = f"{PROJECT_FOLDER}/data"
    os.makedirs(ANALYSIS_FOLDER, exist_ok=True)

    if not output_csv:
        output_csv = f"{ANALYSIS_FOLDER}/image_inventory.csv"

    for root, _, files in os.walk(folder_path):
        for file in files:
            if file.lower().endswith(('.png', '.jpg', '.jpeg', '.tiff', '.bmp', '.gif')):
                # Add image name and path to the list
                image_path = os.path.join(root, file)
                image_list.append([file, image_path])

    # Save the list to a CSV file
    with open(output_csv, 'w', newline='') as csvfile:
        csvwriter = csv.writer(csvfile)
        csvwriter.writerow(['image_name', 'image_path'])  # Write header
        csvwriter.writerows(image_list)  # Write image data

    print(f"List of images saved to {output_csv}")



In [ ]:
if 111 == 111:
    list_images()

List of images saved to /content/drive/MyDrive/Franklin_Project_2024/data/analysis/image_inventory.csv


# (m)Bdd Label Attr

In [ ]:
import json
import csv
from collections import Counter

def bdd_labelling(train_label_file_path=None,
                  val_label_file_path=None,
                  output_label_file_path=None):
    """
    Processes BDD100K label files to extract and summarize information and saves to a CSV file.

    Args:
        train_label_file_path (str): Path to the training JSON file.
        val_label_file_path (str): Path to the validation JSON file.
        output_label_file_path (str): Path to save the output CSV file.
    """
    # Load JSON data from the provided file paths
    if not train_label_file_path:
        train_label_file_path = LABEL_TRAIN_FILE

    if not val_label_file_path:
        val_label_file_path = LABEL_VAL_FILE

    if not output_label_file_path:
        output_label_file_path = f"{ANALYSIS_FOLDER}/bdd100k_labelling.csv"

    with open(train_label_file_path, 'r') as train_file:
        train_data = json.load(train_file)
    with open(val_label_file_path, 'r') as val_file:
        val_data = json.load(val_file)

    # Initialize list to hold processed data
    image_categories = []

    # Function to process a single object and extract necessary details
    def process_object(obj, group):
        """
        Extracts relevant details from a single JSON object.

        Args:
            obj (dict): JSON object representing image data.
            group (str): Indicates whether the data is from training or validation set.

        Returns:
            dict: A dictionary containing extracted and summarized information.
        """
        result = {
            "group": group,
            "name": obj["name"],
            "weather": obj["attributes"].get("weather", "unknown"),
            "scene": obj["attributes"].get("scene", "unknown"),
            "timeofday": obj["attributes"].get("timeofday", "unknown"),
            "categories": ""
        }

        # Extract categories if available
        if "labels" in obj:
            categories = Counter(label["category"] for label in obj["labels"])
            # Format categories as a single string with 'category:count' pairs
            result["categories"] = "; ".join(f"{category}:{count}" for category, count in categories.items())

        return result

    # Process training data
    for obj in train_data:
        image_categories.append(process_object(obj, "train"))

    # Process validation data
    for obj in val_data:
        image_categories.append(process_object(obj, "val"))

    # Define the CSV header fields
    csv_columns = ["group", "name", "weather", "scene", "timeofday", "categories"]

    # Write the processed data to a CSV file
    with open(output_label_file_path, 'w', newline='') as output_file:
        csvwriter = csv.DictWriter(output_file, fieldnames=csv_columns)
        csvwriter.writeheader()
        csvwriter.writerows(image_categories)

    print(f"Processed data saved to {output_label_file_path}")



In [ ]:
if 111 == 111:
    bdd_labelling()


Processed data saved to /content/drive/MyDrive/Franklin_Project_2024/data/analysis/bdd100k_labelling.csv


# (m)Image Master file
- location and attr

In [ ]:
import pandas as pd

def generate_master_file(inventory_file=None, attribute_file=None, output_master_file=None):
    """
    Merges the inventory and attribute files on the image name and saves the result as a master file.

    Args:
        inventory_file (str): Path to the inventory CSV file.
        attribute_file (str): Path to the attribute CSV file.
        output_master_file (str): Path to save the master CSV file.
    """
    if not inventory_file:
        inventory_file = f"{ANALYSIS_FOLDER}/image_inventory.csv"

    if not attribute_file:
        attribute_file = f"{ANALYSIS_FOLDER}/bdd100k_labelling.csv"

    if not output_master_file:
        output_master_file = f"{ANALYSIS_FOLDER}/image_master_file.csv"

    # Load the inventory and attribute files into dataframes
    inventory_df = pd.read_csv(inventory_file)
    attribute_df = pd.read_csv(attribute_file)

    # Merge the two dataframes on the image name columns using a left join
    merged_df = pd.merge(inventory_df, attribute_df, left_on='image_name', right_on='name', how='left')

    # Drop the duplicate 'name' column after the merge
    merged_df = merged_df.drop(columns=['name'])

    # Save the merged dataframe to the output master file
    merged_df.to_csv(output_master_file, index=False)

    print(f"Master file saved to {output_master_file}")



In [ ]:
if 111 == 111:
    generate_master_file()

Master file saved to /content/drive/MyDrive/Franklin_Project_2024/data/analysis/image_master_file.csv


# Calculating RGB

In [ ]:
import pandas as pd
import os
import cv2
import numpy as np
import torch
from torch.utils.data import Dataset, DataLoader
import matplotlib.pyplot as plt
from tqdm import tqdm

# Custom dataset class to handle image loading
class ImageDataset(Dataset):
    def __init__(self, dataframe, root_dir):
        self.dataframe = dataframe
        self.root_dir = root_dir

    def __len__(self):
        return len(self.dataframe)

    def __getitem__(self, idx):
        img_name = os.path.join(self.root_dir, self.dataframe.iloc[idx, 5])
        image = cv2.imread(img_name)

        if image is None:
            print(f"Warning: Failed to read image {img_name}")
            return None, None

        image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
        image = torch.tensor(image).permute(2, 0, 1).float() / 255.0  # Normalize to [0, 1]
        return image, self.dataframe.iloc[idx, 2]

def calculate_mean_std(dataloader, output_csv_file=None, scenario=None):
    scene_rgb_sum = {}
    scene_rgb_count = {}
    scene_rgb_sq_sum = {}
    scene_image_count = {}

    if not scenario:
        scenario = "all"
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

    if not output_csv_file:
        output_csv_file = f"{ANALYSIS_FOLDER}/rgb_mean_std_count_by_{scenario}.csv"

    for images, scenes in tqdm(dataloader, desc="Processing Images"):
        # Filter out None values
        images = [img for img in images if img is not None]
        scenes = [scn for img, scn in zip(images, scenes) if img is not None]

        if len(images) == 0:
            continue

        images = torch.stack(images).to(device)
        for i in range(len(images)):
            image = images[i]
            scene = scenes[i]
            if scene not in scene_rgb_sum:
                scene_rgb_sum[scene] = torch.zeros(3).to(device)
                scene_rgb_count[scene] = 0
                scene_rgb_sq_sum[scene] = torch.zeros(3).to(device)
                scene_image_count[scene] = 0

            scene_rgb_sum[scene] += image.sum(dim=(1, 2))
            scene_rgb_count[scene] += image.shape[1] * image.shape[2]
            scene_rgb_sq_sum[scene] += (image ** 2).sum(dim=(1, 2))
            scene_image_count[scene] += 1

    # Prepare data for saving
    csv_data = []
    for scene in scene_rgb_sum:
        mean = scene_rgb_sum[scene] / scene_rgb_count[scene]
        std = torch.sqrt((scene_rgb_sq_sum[scene] / scene_rgb_count[scene]) - (mean ** 2))
        csv_data.append({
            "scene": scene,
            "mean_r": mean[0].item(),
            "mean_g": mean[1].item(),
            "mean_b": mean[2].item(),
            "std_r": std[0].item(),
            "std_g": std[1].item(),
            "std_b": std[2].item(),
            "count": scene_image_count[scene]
        })

    # Save to CSV
    with open(output_csv_file, 'w', newline='') as csvfile:
        fieldnames = ["scene", "mean_r", "mean_g", "mean_b", "std_r", "std_g", "std_b", "count"]
        writer = csv.DictWriter(csvfile, fieldnames=fieldnames)
        writer.writeheader()
        writer.writerows(csv_data)

    print(f"Results saved to {output_csv_file}")

def plot_results(output_csv_file=None, scenario=None):
    if not scenario:
        scenario = 'all'
    if not output_csv_file:
        output_csv_file = f"{ANALYSIS_FOLDER}/rgb_mean_std_count_by_{scenario}.csv"

    df = pd.read_csv(output_csv_file)

    scenes = df['scene'].tolist()
    means_r = df['mean_r'].tolist()
    means_g = df['mean_g'].tolist()
    means_b = df['mean_b'].tolist()
    stds_r = df['std_r'].tolist()
    stds_g = df['std_g'].tolist()
    stds_b = df['std_b'].tolist()

    fig, axes = plt.subplots(2, 1, figsize=(12, 12))

    # Plot means
    axes[0].bar(scenes, means_r, color='r', alpha=0.6, label='Red')
    axes[0].bar(scenes, means_g, color='g', alpha=0.6, label='Green')
    axes[0].bar(scenes, means_b, color='b', alpha=0.6, label='Blue')
    axes[0].set_title('Mean RGB values by Scene')
    axes[0].set_ylabel('Mean Value')
    axes[0].legend()

    # Plot standard deviations
    axes[1].bar(scenes, stds_r, color='r', alpha=0.6, label='Red')
    axes[1].bar(scenes, stds_g, color='g', alpha=0.6, label='Green')
    axes[1].bar(scenes, stds_b, color='b', alpha=0.6, label='Blue')
    axes[1].set_title('Standard Deviation of RGB values by Scene')
    axes[1].set_ylabel('Standard Deviation')
    axes[1].legend()

    # Set x-axis labels and grid
    for ax in axes:
        ax.set_xticklabels(scenes, rotation=45, ha='right')
        ax.grid(True)

    plt.tight_layout()
    plt.show()

def calculating_rgb_by_scenario(scenario_mapping={}, scenario=None, master_file_path=None):
    # Load the dataset
    if not master_file_path:
        master_file_path = f"{ANALYSIS_FOLDER}/image_master_file.csv"

    df = pd.read_csv(master_file_path)

    # Map scenes to categories
    scene_mapping = {
        'city street': "city street",
        'highway': "highway",
        'residential': "residential",
        'parking lot': "other",
        'undefined': "other",
        'tunnel': "other",
        'gas stations': "other"
    }
    df['scene'] = df['scene'].map(scene_mapping).fillna('other')

    # Create the dataset
    dataset = ImageDataset(df, root_folder)
    dataloader = DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS)

    # Calculate mean and standard deviation
    calculate_mean_std(dataloader)




In [ ]:
if 11 == 111:
    calculating_rgb_by_scenario()

# (m)Prepare Dataset
- this scripts will be used for training as well

**Scenario Mapping E.g.**

In [ ]:

# ['city street', 'highway', 'residential']

scenario_mapping_highway = {
            "scenario_name": "highway_pedestrian",
            "label_columns": {"categories": ["pedestrian"], "timeofday":["-all-"]},
            "scenario_columns": {"weather":["-all-"], "scene":["highway"]}}
scenario_mapping_city = {
            "scenario_name": "city_pedestrian",
            "label_columns": {"categories": ["pedestrian"], "timeofday":["-all-"]},
            "scenario_columns": {"weather":["-all-"], "scene":["city street"]}}
scenario_mapping_residential = {
            "scenario_name": "residential_pedestrian",
            "label_columns": {"categories": ["pedestrian"], "timeofday":["-all-"]},
            "scenario_columns": {"weather":["-all-"], "scene":["residential"]}}
scenario_mapping_overall = {
            "scenario_name": "overall_pedestrian",
            "label_columns": {"categories": ["pedestrian"], "timeofday":["-all-"]},
            "scenario_columns": {"weather":["-all-"], "scene":["-all-"]}}

**Generate Csv for image list for the scenario**

In [ ]:

def balance_and_sample(df, is_binary=False, splits={"train":0.7, "test":0.15, "val": 0.15}):
    df = df[df['inscope'] == 1]

    if is_binary:
        # Count the number of Positive and Negative labels
        positive_count = df[df['label'] == 'Positive'].shape[0]
        negative_count = df[df['label'] == 'Negative'].shape[0]

        # Determine the smaller count
        min_count = min(positive_count, negative_count)

        # Sample the smaller count from both Positive and Negative labels
        positive_samples = df[df['label'] == 'Positive'].sample(n=min_count, random_state=42)
        negative_samples = df[df['label'] == 'Negative'].sample(n=min_count, random_state=42)

        # Combine the sampled rows
        df = pd.concat([positive_samples, negative_samples])

    split = np.random.rand(len(df))
    df['split'] = np.where(split < splits["train"], 'train', np.where(split <splits["train"] +splits["val"], 'val', 'test'))

    return df

def process_csv_labelling(master_file=None, output_scenario_file=None,  scenario_mapping={}, is_binary=False):
    # Load the CSV file
    # categories can't be in scenario_columns
    if not scenario_mapping:
        scenario_mapping = {
            "scenario_name": "scene_pedestrian",
            "label_columns": {"categories": ["pedestrian", "car"], "timeofday":["night", "dawn/dusk"]},
            "scenario_columns": {"weather":["clear", "snowy"], "scene":["highway"]}}

    if not master_file:
        master_file = f"{ANALYSIS_FOLDER}/image_master_file.csv"
    if not output_scenario_file:
        output_scenario_file = f"{ANALYSIS_FOLDER}/image_master_file_{scenario_mapping['scenario_name']}.csv"

    df = pd.read_csv(master_file)

    # Initialize the label and inscope columns

    df['label'] = 'Negative'

    df['inscope'] = 1

    for col in set(scenario_mapping['label_columns'].keys()).union(scenario_mapping['scenario_columns'].keys()):
        df[col] = df[col].fillna("(none)")

    for col, values in scenario_mapping['label_columns'].items():
        if col == "categories":
            df['label'] = df.apply(
                lambda row: "Positive" if all(category in row['categories'] for category in values) else row['label'],
                axis=1
            )
        else:
            if values == ['-all-']:
                pass
            df['label'] = df.apply(
                lambda row: "Positive" if (row['label']=="Positive" and  row[col] in values) else row['label'],
                axis=1
            )
    if not is_binary:
        df = df[df["label"]=="Positive"]
        # Process the label columns
        cols = list(scenario_mapping['label_columns'].keys())
        if "categories" in scenario_mapping["label_columns"].keys():
            category_str = '_'.join(sorted(scenario_mapping["label_columns"]["categories"]))
            cols.remove('categories')
        else:
            category_str =  ''
        for col in cols:
            df['label'] = ('' if category_str=='' else category_str + '_') + df[col]


    # Process the scenario columns
    for col, values in scenario_mapping['scenario_columns'].items():
        if values == ["-all-"]:
            pass
        else:
            df['inscope'] = df.apply(
                lambda row: 1 if row['inscope'] == 1 and row[col] in values else 0,
                axis=1
            )

    # sample and balance it
    df = balance_and_sample(df, is_binary=is_binary)

    # Save to a new CSV file
    df.to_csv(output_scenario_file, index=False)

    return output_scenario_file


In [ ]:
# pd.read_csv('/content/drive/MyDrive/Teaching/AI2024/mini_researches/Proposals/CommonTools/data/analysis/image_master_file_city_pedestrian.csv').groupby('scene').count()

In [ ]:
if 121 == 121:
    for sc in [scenario_mapping_residential, scenario_mapping_highway, scenario_mapping_city, scenario_mapping_overall]:
        output_scenario_file = process_csv_labelling(master_file=None, output_scenario_file=None,  scenario_mapping=sc, is_binary=True)
        print(output_scenario_file)
        # pd.read_csv(f"{ANALYSIS_FOLDER}/image_master_file_scene_pedestrian.csv").head(2)

/content/drive/MyDrive/Franklin_Project_2024/data/analysis/image_master_file_residential_pedestrian.csv
/content/drive/MyDrive/Franklin_Project_2024/data/analysis/image_master_file_highway_pedestrian.csv
/content/drive/MyDrive/Franklin_Project_2024/data/analysis/image_master_file_city_pedestrian.csv
/content/drive/MyDrive/Franklin_Project_2024/data/analysis/image_master_file_overall_pedestrian.csv


In [ ]:
ls /content/drive/MyDrive/Franklin_Project_2024/data/analysis


bdd100k_labelling.csv                  image_master_file_highway_pedestrian.csv
image_inventory.csv                    image_master_file_overall_pedestrian.csv
image_master_file_city_pedestrian.csv  image_master_file_residential_pedestrian.csv
image_master_file.csv


In [ ]:
dfs = pd.read_csv('/content/drive/MyDrive/Franklin_Project_2024/data/analysis/image_inventory.csv')
dfs.shape

(100003, 2)

**Soft custom dataset**

In [ ]:
import torch
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from PIL import Image
import os
import pandas as pd  # Import pandas

class CustomImageDataset(Dataset):
    def __init__(self, scenario, split_name, output_scenario_file=None, transform=None):
        # split_name: train, val, test
        self.transform = transform
        self.scenario = scenario
        self.split_name = split_name

        if not output_scenario_file:
            output_scenario_file = f"{ANALYSIS_FOLDER}/image_master_file_{scenario}.csv"

        # Read the DataFrame and filter based on split_name
        df = pd.read_csv(output_scenario_file)
        df = df[df["split"] == split_name][["image_path", "image_name", "label"]]
        df["image_path"] = df["image_path"] + "/" + df["image_name"]  # Ensure correct path concatenation
        self.image_paths = df[["image_path", "label"]].values

    def __len__(self):
        return len(self.image_paths)

    def __getitem__(self, idx):
        img_path, label = self.image_paths[idx]
        image = Image.open(img_path).convert("RGB")

        if self.transform:
            image = self.transform(image)

        # Use the label directly from the DataFrame
        label = torch.tensor(label, dtype=torch.long)

        return image, label  # Returning both the image and the label


        # dataset = CustomImageDataset(image_dir=image_dir, transform=transform, scene='highway')
        # dataloader = DataLoader(dataset, batch_size=32, shuffle=True)

        # for batch in dataloader:
        #     # Process the batch of filtered images
        #     print(batch.shape)


# Import Libs

In [ ]:
import subprocess
import sys
import importlib

def install_and_import(package):
    try:
        importlib.import_module(package)
    except ImportError:
        subprocess.check_call([sys.executable, "-m", "pip", "install", package])
        importlib.invalidate_caches()
    finally:
        globals()[package] = importlib.import_module(package)

required_packages = [
    "torchmetrics",
    "torchvision",
    "lightning",
    "torchviz"
]

for package in required_packages:
    install_and_import(package)


ModuleNotFoundError: No module named 'lightning'

# Utilities

In [ ]:
def get_image_size(image_path):
    """
    Get the size of an image.
    Args:
        image_path (str): The path to the image file.
    Returns:
        tuple: A tuple containing the width and height of the image in pixels (width, height).
    """
    with Image.open(image_path) as img:
        return img.size  # Returns (width, height)

def get_class_names(is_local=False, data_set_name='bdd', is_experiment=False):
    """
    Retrieves the class names from the training data directory.

    Args:
        is_local (bool): If True, uses local file paths. Default is False.
        use_small_data (bool): If True, uses small dataset paths and sample ratios. Default is True.
        is_experiment (bool): If True, uses the full dataset (i.e., no sampling). Default is False.

    Returns:
        list: A list of class names (subfolder names) in the training data directory.
    """
    # Retrieve data paths and sample ratios
    parameters =  parameters_utility(is_local=is_local,
        data_set_name=data_set_name,
        is_experiment=is_experiment)
    train_path = parameters['train_path']
    # List and return the class names (subfolder names) in the training data directory
    print(f'train_path is {train_path}')
    return os.listdir(train_path)


In [ ]:
files = os.listdir('/content/drive/MyDrive/Franklin_Project_2024/data/bdd100k/bdd100k/images/100k/train')

In [ ]:
len(files)

70000